In [129]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import linear_kernel, cosine_distances, cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [130]:
mm = pd.read_csv('D:\Recommender System\movies_metadata.csv')
mm.head(5)

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\3486657277.py:1: SyntaxWarning: invalid escape sequence '\R'
  mm = pd.read_csv('D:\Recommender System\movies_metadata.csv')
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\3486657277.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  mm = pd.read_csv('D:\Recommender System\movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [131]:
def preprocessing(x):
    if pd.isnull(x):
        return ''
    return ast.literal_eval(x)

def null(x):
    if pd.isnull(x):
        return ''
    return x

mm['belongs_to_collection'] = mm['belongs_to_collection'].apply(preprocessing)
mm['genres'] = mm['genres'].apply(preprocessing)
mm['spoken_languages']=mm['spoken_languages'].apply(preprocessing)
##--------------------------------
mm['id'] = pd.to_numeric(mm['id'], errors='coerce') 
mm = mm.dropna(subset=['id']) 
mm['id'] = mm['id'].astype(int)
mm['budget'] = mm['budget'].astype('int')
##--------------------------------
mm['release_date'] = pd.to_datetime(mm['release_date'], errors='coerce')
mm = mm.sort_values(by=['release_date', 'original_title'], ascending=[0,0])
mm = mm.drop(columns=['homepage','video'])
mm.head()


,adult,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,overview,popularity,...,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
26559,False,"{'id': 87096, 'name': 'Avatar Collection', 'po...",0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,tt1630029,en,Avatar 2,A sequel to Avatar (2009).,6.020055,...,"[{'iso_3166_1': 'US', 'name': 'United States o...",2020-12-16,0.0,0.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",In Production,NaN,Avatar 2,0.0,58.0
38885,False,,12000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,tt0069049,en,The Other Side of the Wind,"Orson Welles' unfinished masterpiece, restored...",0.238154,...,"[{'iso_3166_1': 'IR', 'name': 'Iran'}, {'iso_3...",2018-12-31,0.0,0.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Post Production,NaN,The Other Side of the Wind,0.0,1.0
30402,False,"{'id': 14890, 'name': 'Bad Boys Collection', '...",0,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,tt1502397,en,Bad Boys for Life,The continuing adventures of Miami detectives ...,2.178546,...,"[{'iso_3166_1': 'US', 'name': 'United States o...",2018-11-07,0.0,0.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Planned,NaN,Bad Boys for Life,0.0,12.0
38130,False,,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,tt3906082,en,Mary Shelley,The love affair between poet Percy Shelley and...,3.328261,...,"[{'iso_3166_1': 'IE', 'name': 'Ireland'}, {'is...",2018-04-25,0.0,0.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Post Production,NaN,Mary Shelley,0.0,1.0
44535,False,,0,"[{'id': 18, 'name': 'Drama'}]",412059,tt5613402,en,Mobile Homes,"In forgotten towns along the American border, ...",0.155147,...,"[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...",2018-04-04,0.0,105.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Post Production,NaN,Mobile Homes,0.0,1.0


In [132]:
linksm = pd.read_csv('D:\Recommender System\links_small.csv')
linksm = linksm[linksm['tmdbId'].notnull()]['tmdbId'].astype('int')
linksm.head(5)

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\995298683.py:1: SyntaxWarning: invalid escape sequence '\R'
  linksm = pd.read_csv('D:\Recommender System\links_small.csv')


0      862
1     8844
2    15602
3    31357
4    11862
Name: tmdbId, dtype: int32

In [133]:
linksm_mm = mm['id'].isin(linksm)
linksm_mm = mm[linksm_mm]
linksm_mm.shape

(9099, 22)

In [134]:
linksm_mm['tagline'] = linksm_mm['tagline'].apply(null)
linksm_mm['overview'] = linksm_mm['overview'].apply(null)
linksm_mm['script'] = linksm_mm['tagline'] +' '+ linksm_mm['overview']
linksm_mm['script'] = linksm_mm['script'].apply(null)

C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\932170950.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linksm_mm['tagline'] = linksm_mm['tagline'].apply(null)
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\932170950.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linksm_mm['overview'] = linksm_mm['overview'].apply(null)
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\932170950.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

In [135]:
#TF-IDF
TfIdf_Cal = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=0.0,stop_words='english')
TfIdf_matrix = TfIdf_Cal.fit_transform(linksm_mm['script'])
TfIdf_matrix.shape

(9099, 267986)

In [136]:
linksm_mm = linksm_mm.reset_index()
movie_id_name = pd.Series(linksm_mm.index, index=linksm_mm['title'])
print(movie_id_name)

title
The Lovers and the Despot                                0
The Beatles: Eight Days a Week - The Touring Years       1
Ben-Hur                                                  2
Rustom                                                   3
Hell or High Water                                       4
                                                      ... 
The Immigrant                                         9094
20,000 Leagues Under the Sea                          9095
Intolerance: Love's Struggle Throughout the Ages      9096
The Birth of a Nation                                 9097
A Trip to the Moon                                    9098
Length: 9099, dtype: int64


In [137]:
#Content-Based
def recommender_CB(title,top):
    index = movie_id_name[title]
    cosine_similar_matrix = linear_kernel(TfIdf_matrix, TfIdf_matrix)
    cosine_similar = linear_kernel(TfIdf_matrix[index], TfIdf_matrix).flatten()
    score_similar = pd.Series(cosine_similar, index=movie_id_name.index)
    score_similar = score_similar.drop(title)
    print(cosine_similar_matrix)
    print(cosine_similar)
    return score_similar.nlargest(top).index

In [138]:
recommend_movie=recommender_CB('The Dark Knight',10)
print("Recommend_movie:")
Res=pd.DataFrame({
    'Index': range(1, len(recommend_movie)+1),
    'Title':recommend_movie
})
print(Res.to_string(index=False))

[[1.         0.00193706 0.         ... 0.         0.00467255 0.        ]
 [0.00193706 1.         0.00421165 ... 0.00276998 0.00127864 0.        ]
 [0.         0.00421165 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.00276998 0.         ... 1.         0.00874259 0.        ]
 [0.00467255 0.00127864 0.         ... 0.00874259 1.         0.05407206]
 [0.         0.         0.         ... 0.         0.05407206 1.        ]]
[0.         0.         0.         ... 0.         0.00226419 0.        ]
Recommend_movie:
 Index                                   Title
     1                   The Dark Knight Rises
     2                          Batman Forever
     3                          Batman Returns
     4 Batman: The Dark Knight Returns, Part 2
     5              Batman: Under the Red Hood
     6                                  Batman
     7                        Batman: Year One
     8            Batman: Mask of the Phantasm
     9                                     J

In [139]:
cre = pd.read_csv('D:\Recommender System\credits.csv')  
key=pd.read_csv('D:\Recommender System\keywords.csv')
key['id']=key['id'].astype('int')
cre['id']=cre['id'].astype('int')

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\306566517.py:1: SyntaxWarning: invalid escape sequence '\R'
  cre = pd.read_csv('D:\Recommender System\credits.csv')
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\306566517.py:2: SyntaxWarning: invalid escape sequence '\R'
  key=pd.read_csv('D:\Recommender System\keywords.csv')


In [140]:
mm = mm.merge(cre, on='id')
mm = mm.merge(key, on='id')

meta_mm = mm['id'].isin(linksm)
meta_mm = mm[meta_mm]
meta_mm.head(1)

,adult,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,keywords
975,False,,0,"[{'id': 99, 'name': 'Documentary'}]",373355,tt5278868,en,The Lovers and the Despot,"After the collapse of their glamorous romance,...",0.744523,...,100.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,They were kidnapped by their biggest fan,The Lovers and the Despot,7.0,4.0,"[{'cast_id': 2, 'character': 'Shin Sang-ok', '...","[{'credit_id': '5675d1a5c3a368168b0035c8', 'de...","[{'id': 407, 'name': 'dictator'}, {'id': 1930,..."


In [141]:
meta_mm['crew']=meta_mm['crew'].apply(preprocessing)
meta_mm['keywords']=meta_mm['keywords'].apply(preprocessing)

C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\3674995644.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_mm['crew']=meta_mm['crew'].apply(preprocessing)
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\3674995644.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_mm['keywords']=meta_mm['keywords'].apply(preprocessing)


In [142]:
#KnowLedge-Based
def extract_list_name(temp):
    try:
        if isinstance(temp, str):
            data = ast.literal_eval(temp)
        else:
            data = temp
        return [d['name'] for d in data if 'name' in d]
    except:
        return []
def extract_list_character(temp):
    try:
        if isinstance(temp, str):
            data = ast.literal_eval(temp)
        else:
            data = temp
        return [d['character'] for d in data if 'character' in d]
    except:
        return []
      
#Genres-Key-Cast-Crew: Name
meta_mm['genres_list']=meta_mm['genres'].apply(extract_list_name)
meta_mm['keyw']=meta_mm['keywords'].apply(extract_list_name)
meta_mm['crew_name']=meta_mm['crew'].apply(extract_list_name)
meta_mm['character']=meta_mm['cast'].apply(extract_list_character)
meta_mm.head()

C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\1387419774.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_mm['genres_list']=meta_mm['genres'].apply(extract_list_name)
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\1387419774.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_mm['keyw']=meta_mm['keywords'].apply(extract_list_name)
C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\1387419774.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try

,adult,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,overview,popularity,...,title,vote_average,vote_count,cast,crew,keywords,genres_list,keyw,crew_name,character
975,False,,0,"[{'id': 99, 'name': 'Documentary'}]",373355,tt5278868,en,The Lovers and the Despot,"After the collapse of their glamorous romance,...",0.744523,...,The Lovers and the Despot,7.0,4.0,"[{'cast_id': 2, 'character': 'Shin Sang-ok', '...","[{'credit_id': '5675d1a5c3a368168b0035c8', 'de...","[{'id': 407, 'name': 'dictator'}, {'id': 1930,...",[Documentary],"[dictator, kidnapping, love]","[Robert Cannan, Ross Adam]",[Shin Sang-ok]
976,False,,0,"[{'id': 99, 'name': 'Documentary'}]",373355,tt5278868,en,The Lovers and the Despot,"After the collapse of their glamorous romance,...",0.744523,...,The Lovers and the Despot,7.0,4.0,"[{'cast_id': 2, 'character': 'Shin Sang-ok', '...","[{'credit_id': '5675d1a5c3a368168b0035c8', 'de...","[{'id': 407, 'name': 'dictator'}, {'id': 1930,...",[Documentary],"[dictator, kidnapping, love]","[Robert Cannan, Ross Adam]",[Shin Sang-ok]
1021,False,,0,"[{'id': 99, 'name': 'Documentary'}, {'id': 104...",391698,tt2531318,en,The Beatles: Eight Days a Week - The Touring Y...,"The band stormed Europe in 1963, and, in 1964,...",7.078301,...,The Beatles: Eight Days a Week - The Touring Y...,7.6,92.0,"[{'cast_id': 0, 'character': 'Himself', 'credi...","[{'credit_id': '57057c6cc3a3680dca000285', 'de...","[{'id': 6027, 'name': 'music'}, {'id': 10073, ...","[Documentary, Music]","[music, documentary]","[Brian Grazer, Ron Howard, Ron Howard, Nigel S...","[Himself, Himself, Himself (archive footage), ..."
1172,False,,100000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",271969,tt2638144,en,Ben-Hur,A falsely accused nobleman survives years of s...,11.51021,...,Ben-Hur,5.3,642.0,"[{'cast_id': 1, 'character': 'Judah Ben-Hur', ...","[{'credit_id': '587811b7925141104a00cb16', 'de...","[{'id': 5049, 'name': 'ancient rome'}, {'id': ...","[Adventure, Drama, Action]","[ancient rome, betrayal, vengeance]","[Duncan Henderson, Joni Levin, Timur Bekmambet...","[Judah Ben-Hur, Messala Severus, Jésus Christ,..."
1175,False,,1000000,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...",392572,tt5165344,hi,रुस्तम,"Rustom Pavri, an honourable officer of the Ind...",7.333139,...,Rustom,7.3,25.0,"[{'cast_id': 0, 'character': 'Rustom Pavri', '...","[{'credit_id': '5951baf692514129c4016600', 'de...","[{'id': 10540, 'name': 'bollywood'}]","[Thriller, Romance]",[bollywood],"[Santosh Thundiiayil, Neeraj Pandey, Shital Bh...","[Rustom Pavri, Cynthia Rustom Pavri, Priti Mak..."


In [143]:
def option_choosen(type, Fval):
    valid_columns = ['genres_list', 'keyw', 'crew_name', 'character']
    
    if type not in valid_columns:
        print(f"Loại '{type}' không hợp lệ. Vui lòng chọn trong: {valid_columns}")
        return pd.DataFrame(columns=['title', 'vote_count', 'vote_average'])
    
    meta_mm[type] = meta_mm[type].apply(lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else x))
    df = meta_mm[meta_mm[type].explode().eq(Fval).groupby(level=0).any()]
    
    if df.empty:
        print(f"Không tìm thấy phim nào với '{Fval}' trong '{type}'.")
        return pd.DataFrame(columns=['title', 'vote_count', 'vote_average'])
    
    vote_avg = df[df['vote_average'].notnull()]['vote_average'].astype('float')
    C = vote_avg.mean()
    M = 3000

    specific_data = df[(df['vote_count'] >= M) & (df['vote_count'].notnull()) & (df['vote_average'].notnull())][['title', 'vote_count', 'vote_average']].copy()
    specific_data['vote_count'] = specific_data['vote_count'].astype('int')
    specific_data['vote_average'] = specific_data['vote_average'].astype('float')

    specific_data['wr'] = (
    (specific_data['vote_count'] / (specific_data['vote_count'] + M)) * specific_data['vote_average'] +
    (M / (specific_data['vote_count'] + M)) * C)
    specific_data = specific_data.sort_values('wr', ascending=False).head(250)

    return specific_data

In [144]:

print("\nPhim có từ khóa superhero:")
print(option_choosen('keyw', 'superhero'))



Phim có từ khóa superhero:
                                     title  vote_count  vote_average        wr
15714                      The Dark Knight       12269           8.3  7.867752
9197                 The Dark Knight Rises        9263           7.6  7.233042
9534                          The Avengers       12000           7.4  7.140000
1992                              Deadpool       11444           7.4  7.129992
19734                        Batman Begins        7511           7.5  7.100419
5897   Captain America: The Winter Soldier        5881           7.6  7.093300
15927                             Iron Man        8951           7.4  7.073667
3626               Avengers: Age of Ultron        6908           7.3  6.936657
20415                      The Incredibles        5290           7.4  6.929554
5077                               Birdman        4657           7.4  6.890662
1593            Captain America: Civil War        7462           7.1  6.813248
11134                   

C:\Users\ACER\AppData\Local\Temp\ipykernel_2424\2471436324.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_mm[type] = meta_mm[type].apply(lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else x))


In [145]:
rate = pd.read_csv('D:/Recommender System/ratings_small.csv')
rate['rating'] = rate['rating'].apply(null)
rate['timestamp'] = rate['timestamp'].apply(null)
rate['movieId'] = rate['movieId'].astype('int')
rate['userId'] = rate['userId'].astype('int')
rate['rating'] = rate['rating'].astype('float')
print(rate.head())

   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205


In [146]:
#Cosine
train_data, test_data = train_test_split(rate, test_size=0.2, random_state=42)
ultility_matrix = train_data.pivot(index='userId', columns='movieId', values='rating')
ultility_matrix = ultility_matrix.fillna(0)
cosine_sim = cosine_similarity(ultility_matrix.T)
cosine_simdf = pd.DataFrame(cosine_sim, index=ultility_matrix.columns, columns=ultility_matrix.columns)

print(cosine_simdf)

movieId    1         2         3         4         5         6         7       \
movieId                                                                         
1        1.000000  0.303954  0.240932  0.148076  0.186033  0.346725  0.235109   
2        0.303954  1.000000  0.213363  0.124420  0.171563  0.142269  0.205038   
3        0.240932  0.213363  1.000000  0.147536  0.326404  0.178873  0.306914   
4        0.148076  0.124420  0.147536  1.000000  0.148070  0.049482  0.205545   
5        0.186033  0.171563  0.326404  0.148070  1.000000  0.155262  0.309927   
...           ...       ...       ...       ...       ...       ...       ...   
161830   0.000000  0.089492  0.127343  0.000000  0.000000  0.000000  0.000000   
161918   0.000000  0.089492  0.127343  0.000000  0.000000  0.000000  0.000000   
161944   0.088757  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
162542   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
163949   0.062130  0.000000 

In [149]:
X_train = train_data[['userId', 'movieId']]
Y_train = train_data['rating']

X_test = test_data[['userId', 'movieId']]
Y_test = test_data['rating']

model = XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1)
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
print(f"RMSE is {rmse}")

RMSE is 0.9804587685535319


In [151]:
def collaborative_filtering(userId, model, top):
    complete_matrix = ultility_matrix.columns
    unrated_user = complete_matrix[~complete_matrix.isin(rate[rate['userId']==userId]['movieId'])]
    predict = pd.DataFrame({'userId': userId, 'movieId': unrated_user})
    predict['RaPredict'] = model.predict(predict)

    recommend_CF=predict.sort_values('RaPredict', ascending=False).head(top)
    return recommend_CF[['movieId', 'RaPredict']]

print(collaborative_filtering(1, model, 10))

     movieId  RaPredict
914     1193   4.720052
915     1194   4.720052
916     1196   4.720052
917     1197   4.720052
918     1198   4.720052
919     1199   4.720052
939     1220   4.715631
940     1221   4.715631
938     1219   4.715631
937     1218   4.715631
